# 07 - Liwa Oasis Spatial-Temporal Change & Anomaly Detection

**Arab Youth Space Hackathon 2026** | Team: ليوا الخضراء | Theme: Precision Agriculture & Crop Intelligence

This notebook performs spatial-temporal differencing and anomaly detection over Sentinel-2 scenes in Liwa Oasis to map vegetation stress degradation and soil salinity expansion.

The mechanistic justification for this step is: Statistical differencing of multi-temporal index maps ($\Delta \text{NDVI}$, $\Delta \text{SI}$) isolates degraded date palm sectors for ground-probe deployment.

In [ ]:
# Step 1: Environment Setup & Multi-Temporal Differencing
# The mechanistic justification for this step is: Differencing early vs late season Sentinel-2 scenes computes spatial delta maps.
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import rasterio

def find_files(pattern):
    return sorted(glob.glob(pattern, recursive=True))

s2_b04_files = find_files("/kaggle/input/**/B04.tif") or find_files("../data/**/B04.tif")
s2_b08_files = find_files("/kaggle/input/**/B08.tif") or find_files("../data/**/B08.tif")

if len(s2_b04_files) >= 2 and len(s2_b08_files) >= 2:
    with rasterio.open(s2_b04_files[0]) as r1, rasterio.open(s2_b08_files[0]) as nir1,
         rasterio.open(s2_b04_files[-1]) as r2, rasterio.open(s2_b08_files[-1]) as nir2:
        
        red1, nir1 = r1.read(1).astype(np.float32) / 10000.0, nir1.read(1).astype(np.float32) / 10000.0
        red2, nir2 = r2.read(1).astype(np.float32) / 10000.0, nir2.read(1).astype(np.float32) / 10000.0
        
        min_h = min(red1.shape[0], red2.shape[0])
        min_w = min(red1.shape[1], red2.shape[1])
        
        ndvi1 = (nir1[:min_h, :min_w] - red1[:min_h, :min_w]) / (nir1[:min_h, :min_w] + red1[:min_h, :min_w] + 1e-8)
        ndvi2 = (nir2[:min_h, :min_w] - red2[:min_h, :min_w]) / (nir2[:min_h, :min_w] + red2[:min_h, :min_w] + 1e-8)
        
        delta_ndvi = ndvi2 - ndvi1
        
        print(f"Delta NDVI Range: Min={np.nanmin(delta_ndvi):.4f}, Max={np.nanmax(delta_ndvi):.4f}, Mean={np.nanmean(delta_ndvi):.4f}")
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        im0 = axes[0].imshow(delta_ndvi, cmap='RdYlGn', vmin=-0.3, vmax=0.3)
        axes[0].set_title("Sentinel-2 Delta NDVI (t2 - t1)")
        plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
        
        # Anomaly Mask (Degradation > 2 std dev)
        threshold = np.nanmean(delta_ndvi) - 2 * np.nanstd(delta_ndvi)
        anomaly_mask = delta_ndvi < threshold
        
        im1 = axes[1].imshow(anomaly_mask, cmap='Reds')
        axes[1].set_title(f"Vegetation Stress Anomaly Mask (Delta < {threshold:.3f})")
        plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
        
        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print("Multiple Sentinel-2 scenes not mounted. Skipping change detection...")
